# Day 169 — LangChain Chains & Memory
## Month 10, Day 1 | Google Colab | Groq Free API

---

### Month 10 Overview
| Week | Focus | Days |
|------|-------|------|
| W1 | LangChain Advanced (Chains, Memory, Agents) | 169–172 |
| W2 | MLflow Experiment Tracking | 173–175 |
| W3 | Docker Refresher + Evidently Drift | 176–178 |
| W4 | Prompt Engineering + Month 10 Capstone | 179–180 |

---

### Today: LangChain Chains & Memory
| Task | Topic | Points |
|------|-------|--------|
| T1 | LLMChain + PromptTemplate | 15 |
| T2 | ConversationBufferMemory | 15 |
| T3 | ConversationSummaryMemory + Comparison Table | 20 |
| T4 | SimpleSequentialChain | 15 |
| T5 | SequentialChain (multi-input / multi-output) | 15 |
| ★ | LCEL pipe operator rewrite of T1 | 10★ |
| **Total** | | **80/80 + 10★** |

**Dataset:** ReviewPulse India (600 rows, seed=155) — same as Month 9  
**LLM:** Groq free API → `llama-3.1-8b-instant`  
**Environment:** Google Colab (T4 GPU not required today — CPU is fine)

---

### Why Chains & Memory matter for freelance work
Day 168's RAG pipeline answered single questions. Real client products need:
- **Chains** → connect multiple LLM steps (classify → explain → recommend)
- **Memory** → multi-turn conversation where the LLM remembers earlier context

Together these are the backbone of every AI assistant, chatbot, and automated report you'll build on Upwork.

---
## ⚙️ SECTION 0 — SETUP
### DO NOT MODIFY — run once, restart runtime if asked

In [1]:
# Install dependencies
# langchain-groq brings in langchain-core automatically
!pip install -q langchain langchain-groq langchain-community python-dotenv pandas numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.43 which is incompatible.
langgraph 1.2.5 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.2.43 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.


In [2]:
# --- Groq API key ---
# Option A: Colab Secrets (recommended)
#   Left panel → 🔑 Secrets → add GROQ_API_KEY
# Option B: direct string (never commit to GitHub)

import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"   # <-- replace if needed
    print("⚠️  Using hardcoded key — do not push to GitHub")

✅ GROQ_API_KEY loaded from Colab Secrets


---
## 📦 SECTION 1 — RAW DATA
### ⛔ DO NOT MODIFY ANY CELL IN THIS SECTION

In [3]:
# ============================================================
# RAW DATA — ReviewPulse India (seed=155, n=600)
# LOCKED: do not change seed, n, templates, or probabilities
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(155)
n = 600

sentiments = np.random.choice(
    ['positive', 'negative', 'neutral'], n, p=[0.255, 0.445, 0.30]
)

ratings = []
for s in sentiments:
    if s == 'positive':  ratings.append(np.random.choice([4, 5], p=[0.3, 0.7]))
    elif s == 'negative': ratings.append(np.random.choice([1, 2], p=[0.6, 0.4]))
    else:                 ratings.append(np.random.choice([3, 4], p=[0.8, 0.2]))

hired_again = []
for s in sentiments:
    if s == 'positive':  hired_again.append(np.random.choice([1, 0], p=[0.85, 0.15]))
    elif s == 'negative': hired_again.append(np.random.choice([0, 1], p=[0.90, 0.10]))
    else:                 hired_again.append(np.random.choice([0, 1], p=[0.60, 0.40]))

positive_templates = [
    "Excellent work, delivered on time. Very professional.",
    "Outstanding results, exceeded all expectations.",
    "Great communication and top quality deliverables.",
    "Highly recommend, will definitely hire again.",
    "Perfect execution, understood requirements immediately."
]
negative_templates = [
    "Poor quality work, missed deadlines repeatedly.",
    "Terrible communication, had to ask for updates constantly.",
    "Did not meet requirements, requested multiple revisions.",
    "Very disappointing experience, would not hire again.",
    "Wasted time and money, had to redo the work."
]
neutral_templates = [
    "Decent work overall, met basic requirements.",
    "Average quality, some delays but acceptable.",
    "Okay experience, nothing exceptional to note.",
    "Work was satisfactory, communication could improve.",
    "Met minimum standards, average turnaround time."
]

reviews = []
for s in sentiments:
    if s == 'positive':  reviews.append(np.random.choice(positive_templates))
    elif s == 'negative': reviews.append(np.random.choice(negative_templates))
    else:                 reviews.append(np.random.choice(neutral_templates))

df = pd.DataFrame({
    'review_id': range(1, n + 1),
    'review_text': reviews,
    'sentiment': sentiments,
    'rating': ratings,
    'hired_again': hired_again
})

print(f"Shape: {df.shape}")
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())
print(f"\nOverall hired_again rate: {df['hired_again'].mean():.3f}")
print(f"Positive hired_again rate: {df[df['sentiment']=='positive']['hired_again'].mean():.3f}")
print("\nFirst 3 rows:")
display(df.head(3))

Shape: (600, 5)

Sentiment distribution:
sentiment
negative    266
neutral     180
positive    154
Name: count, dtype: int64

Overall hired_again rate: 0.363
Positive hired_again rate: 0.818

First 3 rows:


,review_id,review_text,sentiment,rating,hired_again
0,1,"Poor quality work, missed deadlines repeatedly.",negative,1,0
1,2,"Average quality, some delays but acceptable.",neutral,4,0
2,3,Great communication and top quality deliverables.,positive,5,1


---
## 📖 SECTION 2 — CONCEPT NOTES

### The LangChain mental model

```
Day 168 (RAG):
  Query → Retriever → LLM → Answer

Day 169 (Chains + Memory):
  Input → [PromptTemplate → LLM] → Output          (LLMChain)
  Input → Chain1 → Chain2 → Output                 (Sequential)
  Message → [Memory + PromptTemplate → LLM]         (Conversation)
```

---

### 2.1 LLMChain + PromptTemplate

**What it is:**  
The simplest LangChain primitive. A `PromptTemplate` has named `{variables}` that get filled at runtime. An `LLMChain` binds a template to an LLM and runs the filled prompt.

```python
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

template = PromptTemplate(
    input_variables=["review"],
    template="Classify this review as positive/negative/neutral.\nReview: {review}\nLabel:"
)
chain = LLMChain(llm=llm, prompt=template)
result = chain.invoke({"review": "Great work!"})
# result["text"] → the LLM's output string
```

**Why it matters (freelance):**  
Every time you need repeatable, parameterised LLM calls — batch classification, report generation, email drafting — `LLMChain` is the right abstraction. It separates the *instruction* (template) from the *data* (variables), so you can swap data without touching the prompt.

---

### 2.2 Memory Types

| Memory Type | How it stores history | Best for | Token cost |
|-------------|----------------------|----------|------------|
| `ConversationBufferMemory` | Appends every message verbatim | Short conversations | Grows linearly |
| `ConversationSummaryMemory` | Summarises older messages via LLM | Long conversations | Fixed (summary length) |
| `ConversationBufferWindowMemory` | Last k messages only | Real-time chat | Fixed (window × avg_tokens) |

**Key concept — memory is injected into the prompt:**
```
System: You are a helpful assistant.
{history}          ← memory injects here
Human: {input}
AI:
```

---

### 2.3 Sequential Chains

**SimpleSequentialChain** — one input → one output → feeds into next chain:
```
review_text → [classify_chain] → "negative" → [action_chain] → "Escalate to support"
```

**SequentialChain** — named inputs/outputs, multiple variables flow through:
```
review_text, rating
  → [classify_chain]  → sentiment
  → [summarise_chain] → summary        (uses review_text + sentiment)
  → [recommend_chain] → recommendation (uses summary + rating)
```

---

### 2.4 LCEL — LangChain Expression Language

The modern (2024+) way to build chains using the `|` pipe operator:
```python
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"review": "Great work!"})
```
LCEL is composable, supports streaming, and is what LangChain now recommends over `LLMChain` for new projects. Both work — you need to know both for client code you'll encounter.

---

### 2.5 The `invoke` vs `run` vs `predict` distinction

| Method | When to use | Returns |
|--------|-------------|--------|
| `.invoke(dict)` | All LangChain v0.1+ chains | Dict with all outputs |
| `.run(str)` | Legacy single-input chains | String only |
| `.predict(key=val)` | Legacy ConversationChain | String only |

**Use `.invoke()` throughout this notebook — it's the current standard.**

---
## ✏️ SECTION 3 — PRACTICE TASKS

Complete all 5 tasks. Each task has:
- `# YOUR CODE HERE` skeleton
- A `print()` or `display()` cell to verify your output
- An NRA insight cell you must fill in

### ⚠️ NRA Rules (same as Month 9):
1. **Number** = exact value from your printed output — never estimated
2. **Reason** = causal mechanism, not outcome description
3. **Action** = specific, committed decision — no hedging words like "would likely" or "could potentially"

In [4]:
!pip install -q langchain==0.2.16 langchain-community==0.2.16 langchain-groq==0.1.9

In [10]:
# --------------------------------------------------------------------
# IMPORTS – All necessary LangChain modules for Tasks 1–5 and Bonus
# --------------------------------------------------------------------
from langchain.prompts import PromptTemplate          # For creating prompt templates
from langchain.chains import LLMChain, SimpleSequentialChain, SequentialChain
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain_core.output_parsers import StrOutputParser  # For LCEL
from langchain_core.prompts import ChatPromptTemplate      # For LCEL
import os
import pandas as pd

---
### ⚙️ LLM Setup (shared by all tasks)

In [6]:
# Shared LLM — used by all tasks
# temperature=0 for reproducibility

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

# Quick smoke test
test_response = llm.invoke("Say 'LangChain ready' and nothing else.")
print("LLM test:", test_response.content)

LLM test: LangChain ready


---
### 📋 TASK 1 — LLMChain + PromptTemplate [15 pts]

**Business goal:** Build a reusable chain that takes any freelancer review text and returns the sentiment label + one-line business reason.

**Steps:**
1. Create a `PromptTemplate` with `input_variables=["review_text"]`
2. The template must instruct the LLM to respond in exactly this format:
   ```
   Sentiment: <positive|negative|neutral>
   Reason: <one sentence>
   ```
3. Create an `LLMChain` binding your template to `llm`
4. Run `.invoke()` on these **3 locked reviews** (use exact text):
   - Review A: `"Excellent work, delivered on time. Very professional."`
   - Review B: `"Wasted time and money, had to redo the work."`
   - Review C: `"Decent work overall, met basic requirements."`
5. Print all 3 outputs
6. Write NRA insight

**Scoring breakdown (15 pts):**
- PromptTemplate correctly defined with `input_variables` [3 pts]
- Template instructs exact `Sentiment: / Reason:` format [3 pts]
- LLMChain created and `.invoke()` called correctly [3 pts]
- All 3 reviews run, outputs printed [3 pts]
- NRA insight (Number + Reason + Action all present) [3 pts]

In [11]:
# --------------------------------------------------------------------
# TASK 1: Build a reusable chain that classifies a review and returns
#         a sentiment label + one-line business reason.
# METHOD: Create a PromptTemplate with a placeholder {review_text}.
#         Instantiate an LLMChain with the template and our LLM.
#         Invoke the chain on three locked reviews.
# --------------------------------------------------------------------

# 1. Define the PromptTemplate with input_variables=["review_text"]
sentiment_template = PromptTemplate(
    input_variables=["review_text"],
    template="""Classify the following freelancer review as positive, negative, or neutral.
Then provide a one-sentence business reason for that classification.
Use exactly this format:

Sentiment: <positive|negative|neutral>
Reason: <one sentence>

Review: {review_text}
"""
)

# 2. Create the LLMChain – this connects the template with our LLM
sentiment_chain = LLMChain(llm=llm, prompt=sentiment_template)

# 3. Locked reviews (exact text from the assignment)
review_A = "Excellent work, delivered on time. Very professional."
review_B = "Wasted time and money, had to redo the work."
review_C = "Decent work overall, met basic requirements."

# 4. Invoke the chain on each review and print the output
for label, review in [("A", review_A), ("B", review_B), ("C", review_C)]:
    result = sentiment_chain.invoke({"review_text": review})   # .invoke expects a dict
    print(f"--- Review {label} ---")
    print(result["text"])   # result is a dict with key "text"
    print()

/tmp/ipykernel_12169/1151384187.py:24: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  sentiment_chain = LLMChain(llm=llm, prompt=sentiment_template)


--- Review A ---
Sentiment: positive
Reason: The freelancer received a positive review due to the client's satisfaction with their work, timeliness, and professionalism.

--- Review B ---
Sentiment: negative
Reason: The freelancer failed to deliver satisfactory work, resulting in wasted resources and additional costs.

--- Review C ---
Sentiment: neutral
Reason: The review does not express strong emotions or praise, but also does not contain any negative comments, indicating a neutral assessment of the freelancer's work.



In [12]:
# NRA Insight (T1)
# Number   → count how many of the 3 reviews were classified correctly vs ground truth
#            (A=positive, B=negative, C=neutral)
# Reason   → WHY does a structured output format (Sentiment:/Reason:) matter for downstream
#            pipeline steps like SimpleSequentialChain?
# Action   → state one specific thing you would add to the PromptTemplate to make batch
#            classification over all 600 reviews more reliable
nra_t1 = """
Number: 3 / 3 reviews classified correctly (A=positive, B=negative, C=neutral)
Reason: A structured output format enables downstream chains to reliably parse the sentiment label via string splitting or regex, avoiding ambiguity that would break subsequent steps.
Action: Add an explicit instruction: "Output only the two lines as shown, with no extra text." to prevent occasional extra commentary and ensure consistency across 600 reviews.
"""
print(nra_t1)


Number: 3 / 3 reviews classified correctly (A=positive, B=negative, C=neutral)
Reason: A structured output format enables downstream chains to reliably parse the sentiment label via string splitting or regex, avoiding ambiguity that would break subsequent steps.
Action: Add an explicit instruction: "Output only the two lines as shown, with no extra text." to prevent occasional extra commentary and ensure consistency across 600 reviews.



---
### 📋 TASK 2 — ConversationBufferMemory [15 pts]

**Business goal:** Build a multi-turn chatbot that remembers what the user said previously — simulating a client asking follow-up questions about their freelancer reviews.

**Steps:**
1. Import and instantiate `ConversationBufferMemory` with `memory_key="chat_history"`, `return_messages=False`
2. Create a `PromptTemplate` that uses `{chat_history}` and `{input}` variables
3. Create an `LLMChain` with `memory=memory_obj`
4. Send **exactly these 3 turns** (use exact text):
   - Turn 1: `"My dataset has 600 freelancer reviews. How many are typically negative in a platform dataset?"`
   - Turn 2: `"In my data, 266 are negative. Is that within a normal range?"`
   - Turn 3: `"Given that negative reviews have only a 9.4% hire-again rate, what action should I recommend to my client?"`
5. Print the LLM response for each turn
6. After turn 3, print `memory_obj.buffer` to show the stored conversation
7. Write NRA insight

**Scoring breakdown (15 pts):**
- Memory instantiated with correct `memory_key` and `return_messages=False` [3 pts]
- Prompt uses `{chat_history}` and `{input}` [3 pts]
- All 3 turns invoked in order [3 pts]
- `memory_obj.buffer` printed after turn 3 [3 pts]
- NRA insight complete [3 pts]

In [13]:
# --------------------------------------------------------------------
# TASK 2: Build a multi‑turn chatbot that remembers conversation history.
# METHOD: Instantiate ConversationBufferMemory with memory_key="chat_history".
#         Create a PromptTemplate that includes {chat_history} and {input}.
#         Wrap in an LLMChain with memory=memory_obj.
#         Send 3 locked turns, print responses and the final buffer.
# --------------------------------------------------------------------

# 1. Instantiate memory – stores raw conversation verbatim
buffer_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=False   # We want a single string, not a list of messages
)

# 2. Prompt template – uses {chat_history} to inject memory and {input} for user query
conv_template = PromptTemplate(
    input_variables=["chat_history", "input"],
    template="""You are a data analytics advisor helping a client understand their freelancer review data.
Previous conversation:
{chat_history}
Client: {input}
Advisor:"""
)

# 3. Create the chain with memory
conv_chain = LLMChain(
    llm=llm,
    prompt=conv_template,
    memory=buffer_memory,
    verbose=False
)

# 4. The three locked turns (exact text)
turns = [
    "My dataset has 600 freelancer reviews. How many are typically negative in a platform dataset?",
    "In my data, 266 are negative. Is that within a normal range?",
    "Given that negative reviews have only a 9.4% hire-again rate, what action should I recommend to my client?"
]

# 5. Run each turn and print the LLM's response
for i, turn in enumerate(turns, 1):
    response = conv_chain.invoke({"input": turn})
    print(f"--- Turn {i} ---")
    print(f"User: {turn}")
    print(f"LLM:  {response['text']}")
    print()

# 6. After all turns, print the raw memory buffer
print("\n=== Memory Buffer ===")
print(buffer_memory.buffer)   # Shows all Human/AI exchanges stored verbatim

--- Turn 1 ---
User: My dataset has 600 freelancer reviews. How many are typically negative in a platform dataset?
LLM:  Based on industry benchmarks, it's common for around 10-20% of freelancer reviews to be negative. This can vary depending on the platform, industry, and other factors. However, as a general rule of thumb, we can expect around 60-120 negative reviews out of 600.

Now, let's dive deeper into your data. Can you tell me more about the platform, the type of freelancers, and the services they offer? This will help us better understand the context and identify any potential red flags or areas for improvement.

--- Turn 2 ---
User: In my data, 266 are negative. Is that within a normal range?
LLM:  With 266 negative reviews out of 600, that's approximately 44.3% of the total reviews. This is significantly higher than the 10-20% range we discussed earlier.

Given the higher-than-expected percentage of negative reviews, it's essential to dig deeper into the data to understand t

In [14]:
# NRA Insight (T2)
# Number   → count the total number of Human/AI turns stored in buffer_memory.buffer
# Reason   → WHY does ConversationBufferMemory become a problem as conversation length grows
#            (think: what happens to token count?)
# Action   → name the specific memory class you would switch to for a 20-turn conversation,
#            and state the LangChain parameter that controls its behaviour
nra_t2 = """
Number: 3 Human + 3 AI turns stored in buffer_memory.buffer
Reason: ConversationBufferMemory appends every message verbatim, so token count grows linearly with each turn, eventually exceeding the LLM's context window and increasing cost/latency.
Action: I would switch to ConversationSummaryMemory for a 20-turn conversation, and tune the `max_token_limit` parameter (or the summarization prompt) to control the summary length and quality.
"""
print(nra_t2)


Number: 3 Human + 3 AI turns stored in buffer_memory.buffer
Reason: ConversationBufferMemory appends every message verbatim, so token count grows linearly with each turn, eventually exceeding the LLM's context window and increasing cost/latency.
Action: I would switch to ConversationSummaryMemory for a 20-turn conversation, and tune the `max_token_limit` parameter (or the summarization prompt) to control the summary length and quality.



---
### 📋 TASK 3 — ConversationSummaryMemory + Comparison Table [20 pts]

**Business goal:** Run the same 3 turns from Task 2 using `ConversationSummaryMemory`. Then build a comparison table showing what each memory type stored.

**Steps:**
1. Import and instantiate `ConversationSummaryMemory` (pass `llm=llm` to it)
2. Build the same prompt and chain structure as Task 2
3. Run the same 3 turns (exact same text)
4. Print `summary_memory.buffer` after all 3 turns
5. Build and print a **comparison DataFrame** with columns:
   `["Memory Type", "After Turn 1", "After Turn 3", "Token Growth", "Best Use Case"]`
   - Fill in `After Turn 1` / `After Turn 3` with a short description of what's stored
   - `Token Growth`: `"Linear"` for Buffer, `"Capped"` for Summary
   - `Best Use Case`: short phrase
6. Write NRA insight

**Scoring breakdown (20 pts):**
- `ConversationSummaryMemory` correctly instantiated with `llm=llm` [4 pts]
- Same 3 turns run correctly [4 pts]
- `summary_memory.buffer` printed and shows a summary (not raw turns) [4 pts]
- Comparison DataFrame built with all 5 columns, correct content [4 pts]
- NRA insight complete and internally consistent [4 pts]

In [15]:
# --------------------------------------------------------------------
# TASK 3: Run the same 3 turns using ConversationSummaryMemory, then
#         build a comparison DataFrame between Buffer and Summary.
# METHOD: Instantiate ConversationSummaryMemory with llm=llm (it needs
#         an LLM to generate summaries). Reuse the same prompt template.
#         Run the same 3 turns, print the summary buffer, then build
#         the DataFrame with descriptive strings.
# --------------------------------------------------------------------

# 1. Instantiate summary memory – it will compress history into a summary
summary_memory = ConversationSummaryMemory(
    llm=llm,                    # Required for summarisation
    memory_key="chat_history",
    return_messages=False
)

# 2. Create a new chain using the same prompt but this memory
summary_chain = LLMChain(
    llm=llm,
    prompt=conv_template,
    memory=summary_memory,
    verbose=False
)

# 3. Run the same 3 turns
for i, turn in enumerate(turns, 1):
    response = summary_chain.invoke({"input": turn})
    print(f"--- Turn {i} ---")
    print(f"LLM: {response['text']}")
    print()

# 4. Print the summary (not raw history)
print("\n=== Summary Memory Buffer ===")
print(summary_memory.buffer)   # Shows a concise summary, not raw turns

# 5. Build comparison DataFrame
comparison_data = {
    "Memory Type":   ["ConversationBufferMemory", "ConversationSummaryMemory"],
    "After Turn 1":  ["Raw full exchange: Human asks about typical negative count, AI responds.",
                      "Summarized first exchange: client asked about typical negativity in datasets."],
    "After Turn 3":  ["Raw full history of 3 Human/AI exchanges, including all text.",
                      "Concise summary: client has 266 negatives, asks about range, then asks for action based on 9.4% hire-again rate."],
    "Token Growth":  ["Linear", "Capped"],
    "Best Use Case": ["Short, focused conversations", "Long-running dialogues with many turns"]
}
comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

--- Turn 1 ---
LLM: Based on industry benchmarks, the average percentage of negative reviews in a freelancer platform dataset can vary depending on several factors such as the platform's quality control measures, freelancer demographics, and the type of services offered. However, a commonly cited benchmark is that around 10-20% of reviews are negative.

In your case, with 600 reviews, this would translate to 60-120 negative reviews. However, it's essential to note that this is just a rough estimate, and the actual percentage of negative reviews in your dataset may be higher or lower.

To get a more accurate understanding of the sentiment in your dataset, I would recommend analyzing the reviews in more detail, including the specific reasons mentioned for negative reviews, the types of services that received negative reviews, and any patterns or trends that emerge from the data.

Would you like to proceed with a more in-depth analysis of the reviews, or would you like to explore other as

,Memory Type,After Turn 1,After Turn 3,Token Growth,Best Use Case
0,ConversationBufferMemory,Raw full exchange: Human asks about typical ne...,"Raw full history of 3 Human/AI exchanges, incl...",Linear,"Short, focused conversations"
1,ConversationSummaryMemory,Summarized first exchange: client asked about ...,"Concise summary: client has 266 negatives, ask...",Capped,Long-running dialogues with many turns


In [22]:
# NRA Insight (T3)
# Number   → count the number of sentences in summary_memory.buffer (read from printed output)
# Reason   → WHY does ConversationSummaryMemory need its own LLM call on every turn
#            (causal mechanism — what is it actually doing?)
# Action   → for a Databrief-style client chatbot with 10+ turns, state which memory type
#            you would use AND the specific parameter you would tune to control quality
nra_t3 = """
Number: summary_memory.buffer contains 13 sentences after 3 turns.
Reason: ConversationSummaryMemory calls the LLM after each turn to compress the entire conversation history into a concise summary, so it requires an additional LLM call per turn to generate the new summary.
Action: For a 10+ turn chatbot, I would use ConversationSummaryMemory with a custom `prompt` to enforce a maximum token limit and adjust `llm` to a cheaper model for summarization to control cost and quality.
"""
print(nra_t3)


Number: summary_memory.buffer contains 13 sentences after 3 turns.
Reason: ConversationSummaryMemory calls the LLM after each turn to compress the entire conversation history into a concise summary, so it requires an additional LLM call per turn to generate the new summary.
Action: For a 10+ turn chatbot, I would use ConversationSummaryMemory with a custom `prompt` to enforce a maximum token limit and adjust `llm` to a cheaper model for summarization to control cost and quality.



---
### 📋 TASK 4 — SimpleSequentialChain [15 pts]

**Business goal:** Build a 2-step automated pipeline:
- Step 1 → classify a review as positive/negative/neutral
- Step 2 → generate a client action recommendation *based on that classification*

**Steps:**
1. Create `chain_1` (classify): takes `{review_text}` → outputs a sentiment label
2. Create `chain_2` (recommend): takes the **output of chain_1** (the label) → outputs a 1-sentence business action
   - Template hint: `"The freelancer received a {sentiment} review. Write one business action the client should take."`
   - **Note:** in `SimpleSequentialChain`, chain_2's input variable name does not matter — it receives chain_1's output automatically
3. Create `SimpleSequentialChain(chains=[chain_1, chain_2], verbose=True)`
4. Run on **2 locked reviews**:
   - Review X: `"Highly recommend, will definitely hire again."`
   - Review Y: `"Did not meet requirements, requested multiple revisions."`
5. Print the final recommendation for each
6. Write NRA insight

**Scoring breakdown (15 pts):**
- chain_1 PromptTemplate and LLMChain correct [3 pts]
- chain_2 PromptTemplate and LLMChain correct [3 pts]
- SimpleSequentialChain assembled correctly [3 pts]
- Both reviews run, verbose output printed [3 pts]
- NRA insight complete [3 pts]

In [17]:
# --------------------------------------------------------------------
# TASK 4: Build a 2‑step pipeline: classify review → generate action
# METHOD: Create two separate LLMChains.
#         chain_1: takes review_text, outputs one-word sentiment.
#         chain_2: takes sentiment (output of chain_1) and outputs action.
#         Assemble with SimpleSequentialChain which auto-feeds outputs.
#         Run on two reviews and print the final recommendation.
# --------------------------------------------------------------------

# 1. chain_1: classify review → sentiment label
classify_prompt = PromptTemplate(
    input_variables=["review_text"],
    template="Classify the following review as exactly one of: positive, negative, or neutral.\nReview: {review_text}\nLabel (one word only):"
)
chain_1 = LLMChain(llm=llm, prompt=classify_prompt)

# 2. chain_2: sentiment label → business action
#    Note: the variable name "sentiment" here will be filled by the output of chain_1
action_prompt = PromptTemplate(
    input_variables=["sentiment"],
    template="The freelancer received a {sentiment} review. Write exactly one business action the client should take (one sentence, start with an action verb):"
)
chain_2 = LLMChain(llm=llm, prompt=action_prompt)

# 3. Assemble SimpleSequentialChain
pipeline = SimpleSequentialChain(
    chains=[chain_1, chain_2],
    verbose=True   # Prints intermediate steps (classification output)
)

# 4. Locked reviews
review_X = "Highly recommend, will definitely hire again."
review_Y = "Did not meet requirements, requested multiple revisions."

# 5. Run and print final recommendation
for label, review in [("X", review_X), ("Y", review_Y)]:
    print(f"\n{'='*50}")
    print(f"Review {label}: {review}")
    output = pipeline.invoke(review)   # SimpleSequentialChain.invoke takes a single string
    print(f"Final recommendation: {output['output']}")


Review X: Highly recommend, will definitely hire again.


> Entering new SimpleSequentialChain chain...
Positive
Post the positive review on the freelancer's social media profiles and website to help increase their visibility and attract more clients.

> Finished chain.
Final recommendation: Post the positive review on the freelancer's social media profiles and website to help increase their visibility and attract more clients.

Review Y: Did not meet requirements, requested multiple revisions.


> Entering new SimpleSequentialChain chain...
Negative
Respond to the negative review by addressing the client's concerns in a professional and apologetic manner, and offering a solution or compromise to resolve the issue.

> Finished chain.
Final recommendation: Respond to the negative review by addressing the client's concerns in a professional and apologetic manner, and offering a solution or compromise to resolve the issue.


In [18]:
# NRA Insight (T4)
# Number   → count how many LLM calls happen per review in this SimpleSequentialChain
# Reason   → WHY is SimpleSequentialChain fragile when chain_1 returns extra words
#            beyond the single sentiment label? (what does chain_2's prompt receive?)
# Action   → name one specific fix to chain_1's prompt that enforces single-word output
#            and prevents the chain_2 prompt from getting polluted text
nra_t4 = """
Number: 2 LLM calls per review in this pipeline (one for classification, one for action).
Reason: SimpleSequentialChain blindly passes the entire output of chain_1 to chain_2. If chain_1 outputs extra text (e.g., "negative\nExplanation: ..."), chain_2 receives that full string and the prompt becomes polluted, causing incorrect or off-topic recommendations.
Action: I would add "Output only the single word label, nothing else." in chain_1's prompt and also use an output parser like StrOutputParser to strip any extra characters.
"""
print(nra_t4)


Number: 2 LLM calls per review in this pipeline (one for classification, one for action).
Reason: SimpleSequentialChain blindly passes the entire output of chain_1 to chain_2. If chain_1 outputs extra text (e.g., "negative
Explanation: ..."), chain_2 receives that full string and the prompt becomes polluted, causing incorrect or off-topic recommendations.
Action: I would add "Output only the single word label, nothing else." in chain_1's prompt and also use an output parser like StrOutputParser to strip any extra characters.



---
### 📋 TASK 5 — SequentialChain (multi-input / multi-output) [15 pts]

**Business goal:** Build a 3-step production pipeline with named variables:
- Step 1 → classify review → `{sentiment}`
- Step 2 → generate a 1-sentence summary using review text + sentiment → `{summary}`
- Step 3 → generate client recommendation using summary + rating → `{recommendation}`

**Steps:**
1. Create 3 LLMChains with named `input_keys` and `output_key`
2. Assemble with `SequentialChain` specifying:
   - `chains=[c1, c2, c3]`
   - `input_variables=["review_text", "rating"]`
   - `output_variables=["sentiment", "summary", "recommendation"]`
   - `verbose=True`
3. Run on this **locked input**:
   ```python
   {"review_text": "Poor quality work, missed deadlines repeatedly.", "rating": 1}
   ```
4. Print `sentiment`, `summary`, and `recommendation` from the result dict
5. Write NRA insight

**Scoring breakdown (15 pts):**
- All 3 chains have correct `output_key` [3 pts]
- `SequentialChain` assembled with correct `input_variables` and `output_variables` [4 pts]
- Locked input used, chain runs without error [4 pts]
- All 3 outputs printed from result dict [2 pts]
- NRA insight complete [2 pts]

In [19]:
# --------------------------------------------------------------------
# TASK 5: Build a 3‑step pipeline with named variables:
#         Step 1: classify → sentiment
#         Step 2: generate summary using review + sentiment → summary
#         Step 3: generate recommendation using summary + rating → recommendation
# METHOD: Create three LLMChains, each with an output_key.
#         Assemble with SequentialChain, specifying input_variables
#         and output_variables. Run on a locked input.
# --------------------------------------------------------------------

# 1. Chain 1: review_text → sentiment
c1_prompt = PromptTemplate(
    input_variables=["review_text"],
    template="Classify as positive, negative, or neutral (one word only).\nReview: {review_text}\nLabel:"
)
c1 = LLMChain(llm=llm, prompt=c1_prompt, output_key="sentiment")

# 2. Chain 2: review_text + sentiment → summary
c2_prompt = PromptTemplate(
    input_variables=["review_text", "sentiment"],
    template="Review: {review_text}\nSentiment: {sentiment}\nWrite a one-sentence business summary of this review:"
)
c2 = LLMChain(llm=llm, prompt=c2_prompt, output_key="summary")

# 3. Chain 3: summary + rating → recommendation
c3_prompt = PromptTemplate(
    input_variables=["summary", "rating"],
    template="Summary: {summary}\nRating: {rating}/5\nWrite one specific client action recommendation (start with an action verb):"
)
c3 = LLMChain(llm=llm, prompt=c3_prompt, output_key="recommendation")

# 4. Assemble SequentialChain
full_pipeline = SequentialChain(
    chains=[c1, c2, c3],
    input_variables=["review_text", "rating"],     # What the pipeline expects
    output_variables=["sentiment", "summary", "recommendation"],   # What it returns
    verbose=True
)

# 5. Locked input
locked_input = {
    "review_text": "Poor quality work, missed deadlines repeatedly.",
    "rating": 1
}

result = full_pipeline.invoke(locked_input)

# 6. Print all three outputs
print("\n=== Pipeline Outputs ===")
print(f"Sentiment:      {result['sentiment']}")
print(f"Summary:        {result['summary']}")
print(f"Recommendation: {result['recommendation']}")



> Entering new SequentialChain chain...

> Finished chain.

=== Pipeline Outputs ===
Sentiment:      Negative.
Summary:        The company has failed to meet expectations, delivering subpar work and consistently missing deadlines, resulting in a negative customer experience.
Recommendation: **Terminate the Contract**

Given the company's consistent failure to meet expectations, deliver subpar work, and miss deadlines, it is recommended to terminate the contract immediately. This action will prevent further negative customer experiences and allow the client to explore alternative vendors that can meet their needs and deliver high-quality services.


In [20]:
# NRA Insight (T5)
# Number   → count total LLM calls for this single input (read from verbose output)
# Reason   → WHY does passing named variables between chains (SequentialChain) give you
#            more control than SimpleSequentialChain? (name the specific technical advantage)
# Action   → describe exactly how you would extend this 3-chain pipeline for Databrief —
#            name a 4th chain and its output_key
nra_t5 = """
Number: 3 LLM calls for 1 input in this SequentialChain.
Reason: SequentialChain allows intermediate outputs to be accessed by name, so later chains can use any combination of earlier results and original inputs, not just the immediate predecessor's entire output. This enables flexible data flow and reuse.
Action: For Databrief, I would add a 4th chain that takes `summary` and `sentiment` to generate a one-sentence "executive recommendation" for the client, with output_key="executive_summary".
"""
print(nra_t5)


Number: 3 LLM calls for 1 input in this SequentialChain.
Reason: SequentialChain allows intermediate outputs to be accessed by name, so later chains can use any combination of earlier results and original inputs, not just the immediate predecessor's entire output. This enables flexible data flow and reuse.
Action: For Databrief, I would add a 4th chain that takes `summary` and `sentiment` to generate a one-sentence "executive recommendation" for the client, with output_key="executive_summary".



---
### ⭐ BONUS — LCEL Pipe Operator [10★]

**Task:** Rebuild T1's sentiment chain using LCEL (LangChain Expression Language).

LCEL chains are built with the `|` pipe operator:
```python
chain = prompt | llm | StrOutputParser()
```

**Steps:**
1. Import `StrOutputParser` from `langchain_core.output_parsers`
2. Rebuild the T1 prompt using `ChatPromptTemplate.from_template(...)` (not `PromptTemplate`)
3. Create the LCEL chain with `|`
4. Run on Review A, B, C from T1 using `.invoke({"review_text": ...})`
5. Print output for all 3
6. Add a comment explaining: *what does `StrOutputParser()` do that `LLMChain` did automatically?*

**Scoring breakdown (10★):**
- `ChatPromptTemplate.from_template()` used (not `PromptTemplate`) [2★]
- LCEL `|` chain assembled correctly [3★]
- All 3 reviews run with `.invoke()` [2★]
- Comment explains `StrOutputParser` role [3★]

In [21]:
# --------------------------------------------------------------------
# BONUS: Rebuild T1's sentiment chain using LCEL (LangChain Expression Language)
# METHOD: Use ChatPromptTemplate.from_template() instead of PromptTemplate.
#         Chain the prompt, the LLM, and StrOutputParser with the | operator.
#         Run on the same 3 reviews and print the outputs.
#         Also include a comment explaining what StrOutputParser does.
# --------------------------------------------------------------------

# 1. Create a ChatPromptTemplate (not a PromptTemplate) – required for LCEL
lcel_prompt = ChatPromptTemplate.from_template(
    """Classify the following freelancer review as positive, negative, or neutral.
Then provide a one-sentence business reason for that classification.
Use exactly this format:

Sentiment: <positive|negative|neutral>
Reason: <one sentence>

Review: {review_text}
"""
)

# 2. Build the LCEL chain using the pipe operator
#    The chain: prompt → LLM → output parser
lcel_chain = lcel_prompt | llm | StrOutputParser()

# 3. Invoke on the same three reviews
for label, review in [("A", review_A), ("B", review_B), ("C", review_C)]:
    output = lcel_chain.invoke({"review_text": review})
    print(f"Review {label}: {output}")

# 4. Explanation of StrOutputParser:
#    In LCEL, the LLM returns a structured object (e.g., ChatMessage).
#    StrOutputParser extracts the string content from that object.
#    In LLMChain, this extraction was done automatically, but in LCEL
#    we need to explicitly add the parser to get a plain string.

Review A: Sentiment: positive
Reason: The freelancer received a positive review due to the client's satisfaction with their work, timeliness, and professionalism.
Review B: Sentiment: negative
Reason: The freelancer failed to deliver satisfactory work, resulting in wasted resources and additional costs.
Review C: Sentiment: neutral
Reason: The review does not express strong emotions or praise, but also does not contain any negative comments, indicating a neutral assessment of the freelancer's work.


---
## 🏆 SECTION 4 — SCORING RUBRIC

| Task | What is graded | Points |
|------|---------------|--------|
| T1 | PromptTemplate + LLMChain + 3 reviews + NRA | 15 |
| T2 | BufferMemory + 3 turns + buffer print + NRA | 15 |
| T3 | SummaryMemory + 3 turns + comparison table + NRA | 20 |
| T4 | SimpleSequentialChain + verbose + 2 reviews + NRA | 15 |
| T5 | SequentialChain (3 chains) + 3 outputs printed + NRA | 15 |
| ★ | LCEL chain + StrOutputParser comment | 10★ |
| **Total** | | **80 + 10★** |

### NRA Deduction Rules (same as Month 9)
| Error | Deduction |
|-------|-----------|
| Number not from printed output (estimated or wrong) | -2 pts |
| Reason describes outcome instead of causal mechanism | -1 pt |
| Action uses hedging language ("would likely", "could") | -1 pt |
| NRA missing entirely | -3 pts |

### Chain Construction Deduction Rules
| Error | Deduction |
|-------|-----------|
| `output_key` missing from T5 chains | -1 pt per chain |
| `memory=` not passed to LLMChain in T2/T3 | -3 pts |
| `input_variables` / `output_variables` wrong in SequentialChain | -4 pts |
| LCEL uses `PromptTemplate` instead of `ChatPromptTemplate` | -2★ |

---

### 🎤 Interview Answer (commit this to memory)

*"What's the difference between `SimpleSequentialChain` and `SequentialChain` in LangChain?"*

> `SimpleSequentialChain` passes one chain's full output as the next chain's single input — clean but inflexible. `SequentialChain` uses named variables so intermediate outputs can be mixed and matched: for example, chain 3 can receive chain 1's `sentiment` and the original `rating` without touching chain 2's `summary`. In production I use `SequentialChain` whenever a later step needs context from an earlier step that isn't immediately adjacent, and I use LCEL for anything new because it supports streaming out of the box."

---

### 📌 GitHub Commit (after submission)
```
feat: Day169 - LangChain Chains & Memory [pending]
```
Repo: `Month10-LangChain-MLflow-Portfolio` (create this repo today)